In [ ]:
from pathlib import Path
import os
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
savedir = Path('OUTPUTS/ImageGen/StableDiffusionXL')
prompts = [
    "ultra-detailed 360 panorama of a modern interior, realistic lighting"
    "A wide panoramic landscape with a bright blue sky, majestic mountains in the background, a calm turquoise sea in the foreground, and lush greenery along the shore. The scene should feel vibrant, sunny, and relaxing, like a holiday postcard photograph, with realistic lighting and high detail."
]
negative_prompt = "lowres, noisy, distorted"
generator = torch.Generator(device=device).manual_seed(42)
lora_repo  = "artificialguybr/360Redmond"

WIDTH  = 1600
HEIGHT = 800
seed = 1234

def process_filename(prompt, seed):
    return str(seed) + "_" + prompt.replace(",","_").replace(" ","_")[:100] + ".png"


### Generation (SD XL)

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

base_model = "stabilityai/stable-diffusion-xl-base-1.0"  # or your current SDXL base

pipe = StableDiffusionXLPipeline.from_pretrained(
    base_model,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None,
    use_safetensors=True
).to(device)

# Load the 360Redmond LoRA
pipe.load_lora_weights(lora_repo)
# pipe.fuse_lora()  # optional
for lora in [False, True]:
    if lora:
        pipe.load_lora_weights(lora_repo)
    else:
        pipe.unload_lora_weights()

    for prompt in prompts:
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=30,
            guidance_scale=5.5,
            generator=generator,
            cross_attention_kwargs={"scale": 0.85},
            width=WIDTH,
            height=HEIGHT,
        ).images[0] 


    full_filename = savedir / "gen" / process_filename(prompt)
    os.makedirs(full_filename.parent, exist_ok=True)
    image.save(full_filename)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

### Inpainting (SD XL)

In [ ]:
import torch
from PIL import Image
from diffusers import StableDiffusionXLInpaintPipeline

# --- Inputs ---
base_model = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"  # SDXL inpaint
dtype = torch.float16 if device == "cuda" else torch.float32
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    base_model,
    torch_dtype=dtype,
    use_safetensors=True,
    variant="fp16" if device == "cuda" else None,
).to(device)

pipe.load_lora_weights(lora_repo)
# pipe.fuse_lora()  # optional

pipe.enable_attention_slicing()

for prompt in prompts:
    image = Image.new("RGB", (512, 512), (127, 127, 127))
    mask  = Image.new("L", (512, 512), 255)  # white=paint, black=keep

    # SDXL often benefits from lower guidance_scale
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        num_inference_steps=30,
        guidance_scale=5.0,
        strength=1.0,
        cross_attention_kwargs={"scale": 0.85},
        generator=generator,
        width=WIDTH,
        height=HEIGHT,
    ).images[0]

    full_filename = savedir / "inpaint" / process_filename(prompt)
    os.makedirs(full_filename.parent, exist_ok=True)
    result.save(full_filename)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


  0%|          | 0/30 [00:00<?, ?it/s]